In [287]:
import os
import torch
import numpy as np
import pandas as pd
import open3d as o3d
from pathlib import Path
import plotly.graph_objects as go
from scipy.spatial import cKDTree
from tqdm import tqdm

run_dir = Path("/app/lab-gatr/runs/labgatr/exp-37")
dataset_dir = Path("/data/Predict-Pneumoperitoneum_LaB-GATr/dataset/raw")
data_idxs = sorted(dataset_dir.glob("*.pt"))
cam_angles = pd.read_csv("/data/Predict-Pneumoperitoneum_LaB-GATr/dataset/camera_angles.csv")
swap_axis = {row["patient"].replace(".pt", ""): row["swap_axis"] for _, row in cam_angles.iterrows()}
reflect_axis = {row["patient"].replace(".pt", ""): row["reflect_axis"] for _, row in cam_angles.iterrows()}

In [288]:
def rmse(pts_a, pts_b):
    return np.linalg.norm(pts_a - pts_b, axis=-1)

def mae(pts_a, pts_b):
    return np.sum(np.abs(pts_a - pts_b), axis=-1) / 3.

def magn(pts_a, pts_b):
    magn_a = np.linalg.norm(pts_a, axis=-1).clip(min=1e-8)
    magn_b = np.linalg.norm(pts_b, axis=-1)
    return np.abs(magn_a - magn_b) / magn_a * 100

def ang(pts_a, pts_b):
    dot = np.sum(pts_b * pts_a, axis=-1)
    denom = np.linalg.norm(pts_b, axis=-1).clip(min=1e-8) * np.linalg.norm(pts_a, axis=-1).clip(min=1e-8)
    cos = np.clip(dot / denom, a_min=-1., a_max=1.)
    return np.arccos(cos) * 180. / np.pi

def medio_lateral_region(marks, pts, swap_axis=False):
    axis = 1 if swap_axis else 0
    min = np.min(pts[:, axis])
    max = np.max(pts[:, axis])
    bounds_region1 = [min * .33, max * .33]
    bounds_region2 = [min * .67, max * .67]
    regions = []
    for mark in marks:
        if mark[axis] > bounds_region1[0] and mark[axis] < bounds_region1[1]:
            regions.append(1)
        elif mark[axis] > bounds_region2[0] and mark[axis] < bounds_region2[1]:
            regions.append(2)
        else:
            regions.append(3)
    return regions

def cranio_caudal_region(marks, pts, swap_axis=False, reflect_axis=False):
    axis = 0 if swap_axis else 1
    reflect = -1. if reflect_axis else 1.
    max = np.max(pts[:, axis] * reflect)
    regions = []
    for mark in marks:
        if mark[axis] * reflect <= 0:
            regions.append(1)
        elif mark[axis] * reflect < max * .33:
            regions.append(2)
        elif mark[axis] * reflect < max * .67:
            regions.append(3)
        else:
            regions.append(4)
    return regions

In [209]:
def add_markers(pts, size=2, opacity=.8, color="red", colorscale=None, showscale=False):
    return go.Scatter3d(
        x = pts[:, 0], y = pts[:, 1], z = -pts[:, 2], mode = "markers",
        marker = {
            "size": size,
            "opacity": opacity,
            "color": color,
            "colorscale": colorscale,
            "showscale": showscale
        })

def add_lines(pts_a, pts_b, width=2, color="black"):
    x, y, z = [], [], []
    for p, q in zip(pts_a, pts_b):
        x += [p[0], q[0], None]
        y += [p[1], q[1], None]
        z += [-p[2], -q[2], None]
    return go.Scatter3d(
        x = x, y = y, z = z, mode = "lines",
        line = {
            "color": color,
            "width": width
        })

def plot_data(data):
    fig = go.Figure(data=data)
    fig.update_layout(
        width = 1200,
        height = 900,
        template = "plotly_white",
        # scene = {"xaxis": {"visible": False}, "yaxis": {"visible": False}, "zaxis": {"visible": False}}, 
        scene_camera = {"eye": {"x": 0., "y": 1.5, "z": 2.0}}
    )
    fig.show()

def plot_abdomen(pts, marks=None):
    data = []
    data.append(add_markers(pts, color=-pts[:, 2], opacity=1., colorscale="Viridis"))
    if marks is not None:
        data.append(add_markers(marks, size=10, opacity=1.))
    plot_data(data)

def plot_abdomen_with_errors(pts, errors, pts_a=None, marks=None):
    data = []
    data.append(add_markers(pts, color=errors, opacity=1., colorscale="RdBu_r", showscale=True))
    if pts_a is not None:
        data.append(add_markers(pts_a, color=-pts_a[:, 2], opacity=1., colorscale="gray"))
    if marks is not None:
        data.append(add_markers(marks, size=10, opacity=1.))
    plot_data(data)

def plot_abdomens(pts_a, pts_b, marks_a=None, marks_b=None):
    data = []
    data.append(add_markers(pts_a, size=1, opacity=.5, color=-pts_a[:, 2], colorscale="Reds"))
    data.append(add_markers(pts_b, size=1, opacity=.2, color=-pts_b[:, 2], colorscale="Greens"))
    if marks_a is not None:
        data.append(add_markers(marks_a, size=5, opacity=1., color="black"))
    if marks_b is not None:
        data.append(add_markers(marks_b, size=5, opacity=1., color="black"))
    if marks_a is not None and marks_b is not None:
        data.append(add_lines(marks_a, marks_b))
    plot_data(data)

def plot_abdomen_regions(pts, marks, regions):
    region_to_color = ["red", "orange", "green", "blue"]
    data = []
    data.append(add_markers(pts, color=-pts[:, 2], opacity=.2, colorscale="Viridis"))
    for mark, region in zip(marks, regions):
        mark = np.expand_dims(mark, axis=0)
        data.append(add_markers(mark, size=5, opacity=1., color=region_to_color[region-1]))
    plot_data(data)

<h3>Single patient</h3>

In [285]:
fold = 0
sample_id = 0

pcd_dir = run_dir / f"fold{fold}" / "vis"
pcd_idxs = sorted([f.stem.replace("pred_", "") for f in pcd_dir.glob("pred_*.ply")])
idx = pcd_idxs[sample_id]
print("Patient:", idx, data_idxs[int(idx)])

pcd_before = np.asarray(o3d.io.read_point_cloud(pcd_dir / f"start_{idx}.ply").points)
pcd_after = np.asarray(o3d.io.read_point_cloud(pcd_dir / f"end_{idx}.ply").points)
pcd_pred = np.asarray(o3d.io.read_point_cloud(pcd_dir / f"pred_{idx}.ply").points)

data = torch.load(data_idxs[int(idx)], weights_only=False)
anns_before = data["annotations_start"].numpy() * 100  # m to cm
anns_after = data["annotations_end"].numpy() * 100  # m to cm

tree = cKDTree(pcd_before)
dist, vertex_idxs = tree.query(anns_before, k=1)
assert np.mean(dist) < 1e-4
anns_pred = pcd_pred[vertex_idxs]

tree = cKDTree(pcd_after)
pred_dists, _ = tree.query(pcd_pred)

print(f"RMSE: {np.mean(rmse(anns_pred, anns_after)):.2f}, MAE: {np.mean(mae(anns_pred, anns_after)):.2f}")

Patient: 0001 /data/Predict-Pneumoperitoneum_LaB-GATr/dataset/raw/24_05_CT+.pt
RMSE: 1.49, MAE: 0.65


In [ ]:
# plot_abdomen_with_errors(pcd_pred, pred_dists, pts_a=pcd_before)
# plot_abdomens(pcd_after, pcd_pred, marks_a=anns_after, marks_b=anns_pred)

ml_regions = medio_lateral_region(anns_before, pcd_before, swap_axis=True)
cc_regions = cranio_caudal_region(anns_before, pcd_before, swap_axis=True, reflect_axis=True)
print("Regions:", ml_regions, "(ML)", cc_regions, "(CC)")
plot_abdomen_regions(pcd_before, anns_before, cc_regions)

<h3>Full dataset</h3>

In [ ]:
rmse_total, mae_total, magn_total, ang_total = [], [], [], []
landmarks = []

for fold in tqdm(np.arange(9)):
    pcd_dir = run_dir / f"fold{fold}" / "vis"
    pcd_idxs = sorted([f.stem.replace("pred_", "") for f in pcd_dir.glob("pred_*.ply")])

    rmse_fold, mae_fold, magn_fold, ang_fold = [], [], [], []
    
    for idx in pcd_idxs:
        patient_id = data_idxs[int(idx)].stem
        pcd_before = np.asarray(o3d.io.read_point_cloud(pcd_dir / f"start_{idx}.ply").points)
        # pcd_after = np.asarray(o3d.io.read_point_cloud(pcd_dir / f"end_{idx}.ply").points)
        pcd_pred = np.asarray(o3d.io.read_point_cloud(pcd_dir / f"pred_{idx}.ply").points)
        
        data = torch.load(data_idxs[int(idx)], weights_only=False)
        anns_before = data["annotations_start"].numpy() * 100  # m to cm
        anns_after = data["annotations_end"].numpy() * 100  # m to cm
        
        tree = cKDTree(pcd_before)
        dist, vertex_idxs = tree.query(anns_before, k=1)
        if np.mean(dist) > 1e-4:
            print(f"! In fold {fold}, sample {idx} the mean anns distance is: {np.mean(dist):.4f}")
        anns_pred = pcd_pred[vertex_idxs]
        
        # tree = cKDTree(pcd_after)
        # pred_dists, _ = tree.query(pcd_pred)

        anns_disp_after = anns_after - anns_before
        anns_disp_pred = anns_pred - anns_before
        
        rmse_sample = rmse(anns_after, anns_pred)
        mae_sample = mae(anns_after, anns_pred)
        magn_sample = magn(anns_disp_after, anns_disp_pred)
        ang_sample = ang(anns_disp_after, anns_disp_pred)
        ml_sample = medio_lateral_region(anns_before, pcd_before, swap_axis=swap_axis[patient_id])
        cc_sample = cranio_caudal_region(anns_before, pcd_before, swap_axis=swap_axis[patient_id], reflect_axis=reflect_axis[patient_id])
        
        for a, b, c, d, e, f in zip(ml_sample, cc_sample, rmse_sample, mae_sample, magn_sample, ang_sample):
            landmarks.append([int(idx), a, b, c, d, e, f])
        
        rmse_fold.append(np.mean(rmse_sample))
        mae_fold.append(np.mean(mae_sample))
        magn_fold.append(np.mean(magn_sample))
        ang_fold.append(np.mean(ang_sample))
        
    rmse_total.append(np.mean(rmse_fold))
    mae_total.append(np.mean(mae_fold))
    magn_total.append(np.mean(magn_fold))
    ang_total.append(np.mean(ang_fold))
    # print(f"{np.mean(rmse_fold):.2f}, {np.mean(mae_fold):.2f}, {np.mean(magn_fold):.2f}, {np.mean(ang_fold):.2f}")

print(f"RMSE: {np.mean(rmse_total):.2f} ({np.std(rmse_total):.2f}), MAE: {np.mean(mae_total):.2f} ({np.std(mae_total):.2f})")
print(f"Magn: {np.mean(magn_total):.2f} ({np.std(magn_total):.2f}), Ang: {np.mean(ang_total):.2f} ({np.std(ang_total):.2f})")

landmarks_df = pd.DataFrame(landmarks, columns=["patient", "medio-lateral", "cranio-caudal", "rmse", "mae", "magn", "angle"])
landmarks_df.head(10)

In [ ]:
import plotly.express as px
fig = px.box(landmarks_df, x="medio-lateral", y="rmse", points="all")
fig.update_layout(title="Medio-lateral regions", xaxis_title="Region", yaxis_title="RMSE")
fig.show()

fig = px.box(landmarks_df, x="cranio-caudal", y="rmse", points="all")
fig.update_layout(title="Cranio-caudal regions", xaxis_title="Region", yaxis_title="RMSE")
fig.show()

<h3>Plot raw data file</h3>

In [ ]:
patient = "24_61_CT-"
data = torch.load(dataset_dir / (patient + ".pt"), weights_only=False)
pcd_before = data["pointcloud_begin"].numpy() * 100
anns_before = data["annotations_start"].numpy() * 100
anns_after = data["annotations_end"].numpy() * 100
plot_abdomen(pcd_before, marks=anns_before)